# Chapter 5: 배치 추론을 단계별로 실행하기
chapter-4 venv 커널을 선택하고 README의 MinIO·MLflow 서버를 먼저 실행한다. 각 셀의 결과를 확인한 뒤 다음 단계로 넘어간다.

In [ ]:
from pathlib import Path
import sys
from datetime import datetime, timezone
root = Path.cwd()
if (root / "chapter-5").is_dir():
    root = root / "chapter-5"
assert (root / "batch_inference.py").exists(), "저장소 루트 또는 chapter-5에서 실행하세요"
sys.path.insert(0, str(root))
import batch_inference as lab
lab.configure()
work = root / "outputs" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
work

## 준비: ID와 시점만 담은 입력
전체 CSV와 피처 원천의 대응을 검사하고 샘플 100명을 MinIO에 올린다. 학습은 다시 하지 않는다.

In [ ]:
manifest = lab.prepare(work, rows=100, model_uri=None)
manifest

## 1. 입력 읽기
모델에 넣을 8개 피처가 아직 없는지 확인한다.

In [ ]:
frame = lab.read(work, manifest)
frame.head()

## 2. 과거 피처 조회
조회 시점과 TTL은 어떤 관계일까? Redis를 사용하고 있을까?

In [ ]:
frame = lab.retrieve(work, manifest)
frame.head()

## 3. 모델 추론
확률과 라벨을 비교한다. 전처리를 다시 학습하지 않는 이유는 무엇일까?

In [ ]:
frame = lab.predict(work, manifest)
frame.head()

## 4. 결과 저장
MinIO Console의 chapter5-batch 버킷에서도 확인한다.

In [ ]:
lab.write(work, manifest)

## 연결 전후 결과 검증
원천 피처로 직접 예측한 결과와 Feast를 거쳐 저장한 결과가 같아야 한다. 모델 성능 평가와는 다르다.

In [ ]:
lab.verify(work, manifest)